<a href="https://colab.research.google.com/github/Prithviraj108/IPL-Cricket-Performance-Dashboard/blob/main/IPL_Phase1_DataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ============================================================
## IPL Cricket Performance Dashboard — Phase 1: Data Cleaning
## Tool: Google Colab (Python + Pandas)
## Author: Prithviraj Shukla
## Data: IPL 2008–2023
## ============================================================

# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Load Data Files from Google Drive
## (Use this instead of manual upload for large files)

In [3]:
import gdown

# Match Data (deliveries — large file)
gdown.download(id="1x76YcQ8rx5m8ulAWAeo1T60orkPWpmob", output="deliveries.csv", quiet=False)

# Match Info Data (matches — small file)
gdown.download(id="1deZWEohXnOkiujYtfCCaZavMsC98EfUJ", output="matches.csv", quiet=False)

matches    = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print(f"matches shape:    {matches.shape}")
print(f"deliveries shape: {deliveries.shape}")

Downloading...
From: https://drive.google.com/uc?id=1x76YcQ8rx5m8ulAWAeo1T60orkPWpmob
To: /content/deliveries.csv
100%|██████████| 36.5M/36.5M [00:00<00:00, 112MB/s]
Downloading...
From: https://drive.google.com/uc?id=1deZWEohXnOkiujYtfCCaZavMsC98EfUJ
To: /content/matches.csv
100%|██████████| 205k/205k [00:00<00:00, 71.9MB/s]


matches shape:    (1024, 18)
deliveries shape: (243817, 23)


## Inspect the Data

In [4]:
print("=== MATCHES ===")
print(matches.head(3).to_string())
print("\nNull counts:\n", matches.isnull().sum())

print("\n\n=== DELIVERIES ===")
print(deliveries.head(3).to_string())
print("\nNull counts:\n", deliveries.isnull().sum())

# Always check actual column names before cleaning
print("\nMatches columns:", matches.columns.tolist())
print("Deliveries columns:", deliveries.columns.tolist())

=== MATCHES ===
        id season       city        date           team1                 team2          toss_winner toss_decision  result  dl_applied               winner  win_by_runs  win_by_wickets player_of_match                                     venue       umpire1    umpire2                umpire3
0  1370353   2023  Ahmedabad  2023/05/29  Gujarat Titans   Chennai Super Kings  Chennai Super Kings         field     D/L           1  Chennai Super Kings            0               5       DP Conway          Narendra Modi Stadium, Ahmedabad   Nitin Menon  RJ Tucker  KN Ananthapadmanabhan
1  1370352   2023  Ahmedabad  2023/05/26  Gujarat Titans        Mumbai Indians       Mumbai Indians         field  normal           0       Gujarat Titans           62               0    Shubman Gill          Narendra Modi Stadium, Ahmedabad   Nitin Menon  RJ Tucker          J Madanagopal
2  1370351   2023    Chennai  2023/05/24  Mumbai Indians  Lucknow Super Giants       Mumbai Indians           bat 

## Clean Matches Table

In [5]:
# Fix date column
matches['date']     = pd.to_datetime(matches['date'], dayfirst=True)
matches['season']   = matches['date'].dt.year
matches['month']    = matches['date'].dt.month
matches['day_name'] = matches['date'].dt.day_name()

# Standardise team names (franchises that changed names over the years)
name_map = {
    'Delhi Daredevils'          : 'Delhi Capitals',
    'Deccan Chargers'           : 'Sunrisers Hyderabad',
    'Rising Pune Supergiants'   : 'Rising Pune Supergiant',
    'Kings XI Punjab'           : 'Punjab Kings',
}
for col in ['team1', 'team2', 'winner', 'toss_winner']:
    matches[col] = matches[col].replace(name_map)

# Fill nulls
matches['winner']          = matches['winner'].fillna('No Result')
matches['player_of_match'] = matches['player_of_match'].fillna('Unknown')

# Drop columns we won't use
matches.drop(columns=['umpire1', 'umpire2'], errors='ignore', inplace=True)

print(f"Cleaned matches: {matches.shape}")
print(matches['season'].value_counts().sort_index())

Cleaned matches: (1024, 18)
season
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
Name: count, dtype: int64


## Clean Deliveries Table

In [11]:
# Fill nulls in dismissal columns
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('not_out')
deliveries['wicket_type']   = deliveries['wicket_type'].fillna('none')

# Rename columns for clarity (handles both old and new dataset column names)
deliveries.rename(columns={
    'striker'       : 'batter', # Map 'striker' to 'batter'
    'runs_off_bat'  : 'batter_runs', # Map 'runs_off_bat' to 'batter_runs'
}, errors='ignore', inplace=True)

# Add is_wicket flag
if 'is_wicket' not in deliveries.columns:
    deliveries['is_wicket'] = (deliveries['player_dismissed'] != 'not_out').astype(int)

# Add boundary flags
deliveries['is_four'] = (deliveries['batter_runs'] == 4).astype(int)
deliveries['is_six']  = (deliveries['batter_runs'] == 6).astype(int)

# Merge season from matches
# Note: handles both 'id' and 'match_id' column naming conventions
match_id_col = 'id' if 'id' in matches.columns else 'match_id'
deliveries = deliveries.merge(
    matches[[match_id_col, 'season']],
    left_on='match_id', right_on=match_id_col, how='left'
)

print(f"Cleaned deliveries: {deliveries.shape}")
print(deliveries[['batter', 'bowler', 'batter_runs', 'is_wicket', 'is_four', 'is_six']].head())

Cleaned deliveries: (243817, 28)
         batter     bowler  batter_runs  is_wicket  is_four  is_six
0       WP Saha  DL Chahar            0          0        0       0
1       WP Saha  DL Chahar            0          0        0       0
2       WP Saha  DL Chahar            1          0        0       0
3  Shubman Gill  DL Chahar            1          0        0       0
4       WP Saha  DL Chahar            1          0        0       0


## Build Batting Summary Table

In [13]:
batting = deliveries.groupby(['batter', 'season_y']).agg(
    matches_played = ('match_id', 'nunique'),
    total_runs      = ('batter_runs', 'sum'),
    balls_faced     = ('batter_runs', 'count'),
    fours          = ('is_four', 'sum'),
    sixes          = ('is_six', 'sum'),
    dismissals     = ('is_wicket', 'sum'),
).reset_index()

# Strike rate
batting['strike_rate'] = (batting['total_runs'] / batting['balls_faced'] * 100).round(2)

# Batting average (runs per dismissal; if never out, use total runs)
batting['batting_avg'] = (
    batting['total_runs'] / batting['dismissals'].replace(0, np.nan)
).round(2).fillna(batting['total_runs'])

print(f"Batting summary: {batting.shape}")
print(batting.sort_values('total_runs', ascending=False).head(10))

Batting summary: (2446, 10)
             batter  season_y  matches_played  total_runs  balls_faced  fours  \
2295        V Kohli      2016              16         973          655     84   
2196   Shubman Gill      2023              17         890          582     85   
847      JC Buttler      2022              17         863          596     84   
491       DA Warner      2016              17         848          579     88   
1062  KS Williamson      2018              17         735          522     64   
1249     MEK Hussey      2013              17         733          580     81   
406        CH Gayle      2012              14         733          472     46   
649    F du Plessis      2023              14         730          483     60   
407        CH Gayle      2013              16         720          484     57   
493       DA Warner      2019              12         692          496     57   

      sixes  dismissals  strike_rate  batting_avg  
2295     38          12     

## Build Bowling Summary Table

In [16]:
# Fill nulls in extras column as they represent 0 runs if not specified
deliveries['extras'] = deliveries['extras'].fillna(0)

# Calculate runs off ball (batter_runs + extras)
deliveries['runs_off_ball'] = deliveries['batter_runs'] + deliveries['extras']

bowling = deliveries.groupby(['bowler', 'season_y']).agg(
    matches_played = ('match_id', 'nunique'),
    balls_bowled   = ('batter_runs', 'count'),
    runs_conceded  = ('runs_off_ball', 'sum'),
    wickets        = ('is_wicket', 'sum'),
).reset_index()

# Economy rate (runs per over)
bowling['economy_rate'] = (bowling['runs_conceded'] / bowling['balls_bowled'] * 6).round(2)

# Bowling average (runs per wicket)
bowling['bowling_avg'] = (
    bowling['runs_conceded'] / bowling['wickets'].replace(0, np.nan)
).round(2)

# Bowling strike rate (balls per wicket)
bowling['bowling_sr'] = (
    bowling['balls_bowled'] / bowling['wickets'].replace(0, np.nan)
).round(2)

print(f"Bowling summary: {bowling.shape}")
print(bowling.sort_values('wickets', ascending=False).head(10))

Bowling summary: (1810, 9)
           bowler  season_y  matches_played  balls_bowled  runs_conceded  \
539      HV Patel      2021              15           361            461   
388      DJ Bravo      2013              18           392            505   
721   JP Faulkner      2013              16           395            436   
757      K Rabada      2020              17           414            565   
1009    MM Sharma      2023              14           268            362   
915      M Morkel      2012              16           389            466   
696     JJ Bumrah      2020              15           384            436   
1499   SL Malinga      2011              16           397            393   
1784    YS Chahal      2022              17           429            536   
1523    SP Narine      2012              15           357            332   

      wickets  economy_rate  bowling_avg  bowling_sr  
539        35          7.66        13.17       10.31  
388        34          7.7

## Build Team Performance Table

In [17]:
# Wins per team per season
wins = matches[matches['winner'] != 'No Result'].groupby(
    ['winner', 'season']
).size().reset_index(name='wins')

# Total matches per team per season
team1 = matches.groupby(['team1', 'season']).size().reset_index(name='count')
team2 = matches.groupby(['team2', 'season']).size().reset_index(name='count')
team1.columns = ['team', 'season', 'count']
team2.columns = ['team', 'season', 'count']
total = pd.concat([team1, team2]).groupby(
    ['team', 'season']
)['count'].sum().reset_index(name='matches_played')

team_perf = total.merge(
    wins, left_on=['team', 'season'], right_on=['winner', 'season'], how='left'
)
team_perf.drop(columns='winner', inplace=True)
team_perf['wins']    = team_perf['wins'].fillna(0).astype(int)
team_perf['losses']  = team_perf['matches_played'] - team_perf['wins']
team_perf['win_pct'] = (team_perf['wins'] / team_perf['matches_played'] * 100).round(1)

print(f"Team performance: {team_perf.shape}")
print(team_perf.sort_values(['season', 'win_pct'], ascending=[True, False]).head(12))

Team performance: (136, 6)
                            team  season  matches_played  wins  losses  \
88              Rajasthan Royals    2008              16    13       3   
72                  Punjab Kings    2008              15    10       5   
0            Chennai Super Kings    2008              16     9       7   
14                Delhi Capitals    2008              14     7       7   
53                Mumbai Indians    2008              14     7       7   
35         Kolkata Knight Riders    2008              13     6       7   
104  Royal Challengers Bangalore    2008              14     4      10   
120          Sunrisers Hyderabad    2008              14     2      12   
15                Delhi Capitals    2009              15    10       5   
1            Chennai Super Kings    2009              14     8       6   
105  Royal Challengers Bangalore    2009              16     9       7   
121          Sunrisers Hyderabad    2009              16     9       7   

     win_p

## Export All Tables to SQLite

In [18]:
conn = sqlite3.connect('ipl_database.db')

matches.to_sql('matches',           conn, if_exists='replace', index=False)
deliveries.to_sql('deliveries',     conn, if_exists='replace', index=False)
batting.to_sql('batting_stats',     conn, if_exists='replace', index=False)
bowling.to_sql('bowling_stats',     conn, if_exists='replace', index=False)
team_perf.to_sql('team_performance',conn, if_exists='replace', index=False)

conn.close()
print("All 5 tables written to ipl_database.db")

# Download the .db file to your computer
from google.colab import files
files.download('ipl_database.db')

All 5 tables written to ipl_database.db


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Sanity Check

In [19]:
conn = sqlite3.connect('ipl_database.db')

for table in ['matches', 'deliveries', 'batting_stats', 'bowling_stats', 'team_performance']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    print(f"  {table:25s}: {count:,} rows")

conn.close()
print("\nPhase 1 complete. Ready for Phase 2 — SQL!")

  matches                  : 1,024 rows
  deliveries               : 243,817 rows
  batting_stats            : 2,446 rows
  bowling_stats            : 1,810 rows
  team_performance         : 136 rows

Phase 1 complete. Ready for Phase 2 — SQL!


## SQL Queries (Phase 2)

In [20]:
conn = sqlite3.connect('ipl_database.db')
print("Connected to ipl_database.db")

Connected to ipl_database.db


In [21]:
# Query 1 — Top 10 Run Scorers
q1 = pd.read_sql("""
    SELECT
        batter,
        SUM(total_runs)             AS total_runs,
        SUM(matches_played)         AS matches,
        ROUND(AVG(batting_avg), 2)  AS avg_batting_avg,
        ROUND(AVG(strike_rate), 2)  AS avg_strike_rate,
        SUM(fours)                  AS total_fours,
        SUM(sixes)                  AS total_sixes
    FROM batting_stats
    GROUP BY batter
    ORDER BY total_runs DESC
    LIMIT 10
""", conn)
print("=== TOP 10 RUN SCORERS ===")
print(q1.to_string(index=False))

=== TOP 10 RUN SCORERS ===
        batter  total_runs  matches  avg_batting_avg  avg_strike_rate  total_fours  total_sixes
       V Kohli        7273      229            35.81           124.11          646          235
      S Dhawan        6617      216            33.75           120.70          750          149
     DA Warner        6399      176            40.12           134.18          646          226
     RG Sharma        6213      237            28.93           126.81          554          258
      SK Raina        5536      200            33.72           131.75          506          204
AB de Villiers        5181      170            40.23           143.01          414          253
      MS Dhoni        5082      217            36.82           131.42          349          239
      CH Gayle        4997      141            38.48           137.20          408          359
    RV Uthappa        4954      197            26.86           126.43          481          182
    KD Karthi

In [22]:
# Query 2 — Best Strike Rate (min 500 balls)
q2 = pd.read_sql("""
    SELECT
        batter,
        SUM(total_runs)                                       AS total_runs,
        SUM(balls_faced)                                      AS balls_faced,
        ROUND(SUM(total_runs) * 100.0 / SUM(balls_faced), 2) AS career_strike_rate,
        SUM(sixes)                                            AS total_sixes
    FROM batting_stats
    GROUP BY batter
    HAVING SUM(balls_faced) >= 500
    ORDER BY career_strike_rate DESC
    LIMIT 10
""", conn)
print("\n=== BEST STRIKE RATE (min 500 balls) ===")
print(q2.to_string(index=False))


=== BEST STRIKE RATE (min 500 balls) ===
        batter  total_runs  balls_faced  career_strike_rate  total_sixes
    AD Russell        2266         1374              164.92          193
LS Livingstone         828          527              157.12           59
    GJ Maxwell        2720         1796              151.45          158
     SP Narine        1046          692              151.16           64
      N Pooran        1270          847              149.94           91
      V Sehwag        2728         1833              148.83          106
AB de Villiers        5181         3487              148.58          253
   YBK Jaiswal        1172          807              145.23           48
    SO Hetmyer        1130          782              144.50           75
    JC Buttler        3224         2253              143.10          149


In [23]:
# Query 3 — Top 10 Wicket Takers
q3 = pd.read_sql("""
    SELECT
        bowler,
        SUM(wickets)                                                  AS total_wickets,
        SUM(matches_played)                                           AS matches,
        ROUND(SUM(runs_conceded) * 6.0 / SUM(balls_bowled), 2)       AS career_economy,
        ROUND(SUM(runs_conceded) * 1.0 / NULLIF(SUM(wickets), 0), 2) AS bowling_avg
    FROM bowling_stats
    GROUP BY bowler
    ORDER BY total_wickets DESC
    LIMIT 10
""", conn)
print("\n=== TOP 10 WICKET TAKERS ===")
print(q3.to_string(index=False))


=== TOP 10 WICKET TAKERS ===
    bowler  total_wickets  matches  career_economy  bowling_avg
  DJ Bravo            207      158            8.08        21.43
 YS Chahal            194      144            7.59        21.30
  R Ashwin            189      194            6.88        26.28
SL Malinga            188      122            7.03        18.54
 PP Chawla            188      180            7.94        25.89
   B Kumar            184      160            7.28        24.43
 SP Narine            182      161            6.76        23.60
  A Mishra            182      161            7.30        22.93
 RA Jadeja            161      197            7.56        28.24
 JJ Bumrah            161      120            7.35        21.73


In [24]:
# Query 4 — Best Economy Rate (min 200 balls)
q4 = pd.read_sql("""
    SELECT
        bowler,
        SUM(wickets)                                                   AS total_wickets,
        SUM(balls_bowled)                                              AS balls_bowled,
        ROUND(SUM(runs_conceded) * 6.0 / SUM(balls_bowled), 2)        AS career_economy,
        ROUND(SUM(balls_bowled) * 1.0 / NULLIF(SUM(wickets), 0), 2)   AS bowling_strike_rate
    FROM bowling_stats
    GROUP BY bowler
    HAVING SUM(balls_bowled) >= 200
    ORDER BY career_economy ASC
    LIMIT 10
""", conn)
print("\n=== BEST ECONOMY RATE (min 200 balls) ===")
print(q4.to_string(index=False))


=== BEST ECONOMY RATE (min 200 balls) ===
        bowler  total_wickets  balls_bowled  career_economy  bowling_strike_rate
 Sohail Tanvir             24           265            6.23                11.04
    A Chandila             11           234            6.28                21.27
    SM Pollock             13           280            6.58                21.54
      A Kumble             49           983            6.65                20.06
    GD McGrath             14           329            6.67                23.50
M Muralitharan             67          1581            6.70                23.60
       J Yadav              9           398            6.74                44.22
   Rashid Khan            147          2639            6.75                17.95
     SP Narine            182          3812            6.76                20.95
      DW Steyn            105          2282            6.79                21.73


In [25]:
# Query 5 — Toss Impact Analysis
q5 = pd.read_sql("""
    SELECT
        toss_decision,
        COUNT(*)                                                         AS total_matches,
        SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END)           AS toss_winner_won,
        ROUND(SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 1)                                     AS win_pct
    FROM matches
    WHERE winner != 'No Result'
    GROUP BY toss_decision
""", conn)
print("\n=== TOSS IMPACT ANALYSIS ===")
print(q5.to_string(index=False))


=== TOSS IMPACT ANALYSIS ===
toss_decision  total_matches  toss_winner_won  win_pct
          bat            365              167     45.8
        field            640              350     54.7


In [26]:
# Query 6 — Team Win % by Season
q6 = pd.read_sql("""
    SELECT
        team,
        season,
        matches_played,
        wins,
        losses,
        win_pct
    FROM team_performance
    ORDER BY season DESC, win_pct DESC
""", conn)
print("\n=== TEAM WIN % BY SEASON ===")
print(q6.to_string(index=False))


=== TEAM WIN % BY SEASON ===
                       team  season  matches_played  wins  losses  win_pct
             Gujarat Titans    2023              17    11       6     64.7
        Chennai Super Kings    2023              16    10       6     62.5
             Mumbai Indians    2023              16     9       7     56.2
       Lucknow Super Giants    2023              15     8       7     53.3
           Rajasthan Royals    2023              14     7       7     50.0
Royal Challengers Bangalore    2023              14     7       7     50.0
      Kolkata Knight Riders    2023              14     6       8     42.9
               Punjab Kings    2023              14     6       8     42.9
             Delhi Capitals    2023              14     5       9     35.7
        Sunrisers Hyderabad    2023              14     4      10     28.6
             Gujarat Titans    2022              16    12       4     75.0
       Lucknow Super Giants    2022              15     9       6     

In [27]:
# Query 7 — Best Venues for Chasing (min 10 matches)
q7 = pd.read_sql("""
    SELECT
        venue,
        COUNT(*)                                                         AS total_matches,
        SUM(CASE WHEN win_by_runs > 0 THEN 1 ELSE 0 END)                AS batting_first_wins,
        SUM(CASE WHEN win_by_wickets > 0 THEN 1 ELSE 0 END)             AS chasing_wins,
        ROUND(SUM(CASE WHEN win_by_wickets > 0 THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 1)                                     AS chase_win_pct
    FROM matches
    WHERE winner != 'No Result'
    GROUP BY venue
    HAVING total_matches >= 10
    ORDER BY chase_win_pct DESC
    LIMIT 10
""", conn)
print("\n=== BEST VENUES FOR CHASING ===")
print(q7.to_string(index=False))


=== BEST VENUES FOR CHASING ===
                                       venue  total_matches  batting_first_wins  chasing_wins  chase_win_pct
                      Sawai Mansingh Stadium             47                  15            32           68.1
                             SuperSport Park             12                   4             8           66.7
                     Sharjah Cricket Stadium             28                  10            18           64.3
     Maharashtra Cricket Association Stadium             22                   8            14           63.6
                 Arun Jaitley Stadium, Delhi             11                   4             7           63.6
                    Wankhede Stadium, Mumbai             38                  14            24           63.2
Punjab Cricket Association IS Bindra Stadium             10                   4             6           60.0
                        Sheikh Zayed Stadium             27                  11            16  

In [28]:
# Query 8 — Most Player of the Match Awards
q8 = pd.read_sql("""
    SELECT
        player_of_match,
        COUNT(*)               AS potm_awards,
        COUNT(DISTINCT season) AS seasons_active
    FROM matches
    WHERE player_of_match != 'Unknown'
    GROUP BY player_of_match
    ORDER BY potm_awards DESC
    LIMIT 10
""", conn)
print("\n=== MOST PLAYER OF THE MATCH AWARDS ===")
print(q8.to_string(index=False))


=== MOST PLAYER OF THE MATCH AWARDS ===
player_of_match  potm_awards  seasons_active
 AB de Villiers           25              11
       CH Gayle           22               9
      RG Sharma           19              12
      DA Warner           18              10
       MS Dhoni           17              11
      YK Pathan           16               8
        V Kohli           16               8
      SR Watson           16               9
       SK Raina           14              10
      RA Jadeja           14               8


## Export All Query Results to CSV

In [29]:
q1.to_csv('top_run_scorers.csv',    index=False)
q2.to_csv('best_strike_rates.csv',  index=False)
q3.to_csv('top_wicket_takers.csv',  index=False)
q4.to_csv('best_economy_rates.csv', index=False)
q5.to_csv('toss_impact.csv',        index=False)
q6.to_csv('team_win_by_season.csv', index=False)
q7.to_csv('venue_chase_stats.csv',  index=False)
q8.to_csv('player_of_match.csv',    index=False)

print("All 8 CSVs exported successfully!")

All 8 CSVs exported successfully!


In [30]:
# Download all CSVs
csv_files = [
    'top_run_scorers.csv',
    'best_strike_rates.csv',
    'top_wicket_takers.csv',
    'best_economy_rates.csv',
    'toss_impact.csv',
    'team_win_by_season.csv',
    'venue_chase_stats.csv',
    'player_of_match.csv',
]
for f in csv_files:
    files.download(f)
    print(f"Downloaded: {f}")

conn.close()
print("\nAll done! Ready for Phase 3 — Power BI")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: top_run_scorers.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded: venue_chase_stats.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: player_of_match.csv

All done! Ready for Phase 3 — Power BI
